# Train the CamDetec rodent model (free)

Trains a **YOLOv8** rodent detector on **your three Roboflow datasets**, merges
them into one clean single-class (`rodent`) set, and exports to **TensorFlow.js**
— producing a `model.json` that drops straight into CamDetec's pest-model slot.

Runs free on a **Colab T4 GPU** (Runtime → Change runtime type → **T4 GPU**).
Nothing to click in Roboflow — the notebook generates dataset versions itself.

**One-time setup:** add your Roboflow key as a Colab secret (next section).

In [ ]:
%pip install -q ultralytics roboflow pyyaml

## 1. GPU check

Should print `cuda … True`. If not: Runtime → Change runtime type → **T4 GPU**.

In [ ]:
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (switch to a GPU runtime)")

## 2. Roboflow API key (kept out of the notebook)

In Colab's left sidebar click the **🔑 key icon → Add new secret**, name it
`ROBOFLOW_API_KEY`, paste your **Private API Key** (app.roboflow.com → Settings →
API Keys), and enable notebook access. The key is never written into this file.

In [ ]:
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    ROBOFLOW_API_KEY = input('Roboflow Private API Key: ').strip()
assert ROBOFLOW_API_KEY, "no API key"
print("key loaded")

## 3. Download your three rodent datasets

Keyed to your Roboflow workspace. The notebook generates a version for each (at
640px) if one doesn't exist, then downloads it in YOLOv8 format. To drop a
dataset, comment it out of `PROJECTS`.

In [ ]:
from roboflow import Roboflow

WORKSPACE = "rahils-workspace-jkbqh"
PROJECTS = [
    "rodent-cdscb-rc3pn",            # "Rodent"           ~818 imgs
    "rodent-detection-qodn4-19ued",  # "rodent detection" ~4980 imgs (cleanest)
    "rodent_detection-oljub",        # "Rodent_detection" ~4148 imgs
]
GEN_SETTINGS = {
    "augmentation": {},
    "preprocessing": {"auto-orient": True, "resize": {"width": 640, "height": 640, "format": "Stretch to"}},
}

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
dataset_dirs = []
for i, pid in enumerate(PROJECTS):
    proj = rf.workspace(WORKSPACE).project(pid)
    try:
        v = proj.generate_version(settings=GEN_SETTINGS)
    except Exception as e:
        print(f"[{pid}] generate_version failed ({e}); trying version 1")
        v = 1
    ds = proj.version(v).download("yolov8", location=f"/content/ds_{i}")
    dataset_dirs.append(ds.location)
    print(f"[{pid}] version {v} -> {ds.location}")
print("downloaded:", dataset_dirs)

## 4. Merge + collapse to a single `rodent` class

Your datasets have mixed/messy class names (a stray `0`, a leaked dataset-name
class). We're rodent-focused, so every box becomes one class: **rodent**. This
sidesteps all the label-name noise.

In [ ]:
import os, glob, shutil, yaml

MERGED = "/content/rodent_merged"
SPLITS = ["train", "valid", "test"]
for s in SPLITS:
    os.makedirs(f"{MERGED}/{s}/images", exist_ok=True)
    os.makedirs(f"{MERGED}/{s}/labels", exist_ok=True)

counts = {s: 0 for s in SPLITS}
for i, loc in enumerate(dataset_dirs):
    for s in SPLITS:
        img_dir = f"{loc}/{s}/images"
        lbl_dir = f"{loc}/{s}/labels"
        if not os.path.isdir(img_dir):
            continue
        for img in os.listdir(img_dir):
            base, ext = os.path.splitext(img)
            new = f"d{i}_{base}"
            shutil.copy(f"{img_dir}/{img}", f"{MERGED}/{s}/images/{new}{ext}")
            lp = f"{lbl_dir}/{base}.txt"
            if os.path.exists(lp):
                out = []
                for ln in open(lp).read().strip().splitlines():
                    p = ln.split()
                    if len(p) >= 5:
                        p[0] = "0"          # single class -> rodent
                        out.append(" ".join(p))
                open(f"{MERGED}/{s}/labels/{new}.txt", "w").write("\n".join(out))
            counts[s] += 1

data = {"path": MERGED, "train": "train/images", "val": "valid/images",
        "test": "test/images", "nc": 1, "names": ["rodent"]}
yaml.safe_dump(data, open(f"{MERGED}/data.yaml", "w"))
DATA_YAML = f"{MERGED}/data.yaml"
print("merged image counts:", counts)
print("DATA_YAML:", DATA_YAML)

## 5. Train

`yolov8n` (nano) — small enough to run in the browser. ~100 epochs on a T4 is
typically 30–90 min for ~10k images; early-stops if it plateaus.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,          # MUST stay 640 — the app's YoloTfjsDetector expects imgsz=640
    batch=16,
    patience=25,
    name="camdetec_rodent",
)

## 6. Quick check — mAP50 above ~0.6 is a good rodent start.

In [ ]:
m = model.val()
print("mAP50:", round(float(m.box.map50), 3), "| mAP50-95:", round(float(m.box.map), 3))

## 7. Export to TensorFlow.js (imgsz 640, no embedded NMS — the app does NMS)

In [ ]:
export_path = model.export(format="tfjs", imgsz=640)
print("TF.js model at:", export_path)

## 8. Write labels.json + download the bundle

In [ ]:
import json, os, glob, shutil

web = str(export_path)
if not os.path.isdir(web):
    c = glob.glob("runs/detect/camdetec_rodent*/weights/*_web_model")
    web = c[-1] if c else web
json.dump(["rodent"], open(os.path.join(web, "labels.json"), "w"))
print("web model:", web, "->", os.listdir(web))

shutil.make_archive("rodent_model", "zip", web)
try:
    from google.colab import files
    files.download("rodent_model.zip")
except Exception:
    print("Grab rodent_model.zip from the file browser.")

## 9. Install it in CamDetec

1. **Unzip** `rodent_model.zip` → `model.json`, `group1-shard*.bin`, `labels.json`.
2. Copy all into the repo at `public/models/pest/`.
3. Redeploy: `vercel --prod` (or commit + push).
4. In the app: **Settings → Object model → "Pest model · YOLOv8"**. It now detects
   `rodent`. The pest map, tracking, and trap suggestions all key off it.

**If accuracy is low:** dataset 3 (`rodent_detection-oljub`) has noisy labels —
try commenting it out of `PROJECTS` and retraining on the two clean sets. More
frames from your actual camera angle (and night/IR) help most. Retrain, re-export,
replace the files in `public/models/pest/`.